# 第5回　テキスト分析の基礎

文字列の使い方は第2回講義でやっているのでそれを参照、ここでは課題の取り組みについて記載するに留める

### Q1

n次元ベクトル空間における、任意の2つのベクトル  

$\mathbf{x} = (x_1, x_2, \ldots, x_n)$, $\mathbf{y} = (y_1, y_2, \ldots, y_n)$ の間のcos類似度 $\cos(X, Y)$は以下のように定義されます。

$$
\cos(X, Y) = \frac{\mathbf{x} \cdot \mathbf{y}}{\|\mathbf{x}\|_2 \|\mathbf{y}\|_2}
= \frac{\sum_{i=1}^{n} x_i y_i}{\sqrt{\sum_{i=1}^{n} x_i^2} \sqrt{\sum_{i=1}^{n} y_i^2}}
$$

入力ベクトル $\mathbf{x}, \mathbf{y}$ をそれぞれ NumPy の配列として引数で受け取り、  
それらのベクトル間のcos類似度を計算して返す関数 `cos_sim` を完成させてください。

In [1]:
import numpy as np

In [2]:
def cos_sim(vec1, vec2):
  # linalg.norm()で
  calc =  np.dot(vec1,vec2) / np.linalg.norm(vec1) * np.linalg.norm(vec2)
  return calc

x = [1,2,3,4,5]
y = [2,4,6,8,10]

print(cos_sim(x,y))

220.0


### Q2
"course_list.csv" ファイルには以下のように各行に講義名のテキストデータが含まれています。以下では、このファイルを読み込み、各講義のベクトルを作成し、講義間の類似度を求めるコードを実装します。

``` 
## course_list.csvファイル
行動神経科学
認知心理学
性格心理学
スポーツ栄養学
グローバル教養特別演習Ⅰ(13)
認知行動障害論
スポーツバイオメカニクス
...
``` 

まず必要なモジュールをimportします。

In [3]:
import json
import csv
import numpy as np
import pandas as pd

### Q2.1
`course_list.csv`ファイルを1行ずつ順番に読み込み、その各行を要素とするリストを作成して返す course list 関数を完成さてください。作成されたリストは変数 courses で受け取ります。以降の処理では、リスト courses のインデックスをその要素(講義名)のIDとして扱います。

In [4]:
def course_list ():
  courses = []
  with open("data/course_list.csv") as f:
    reader = csv.reader(f)

    for row in reader:
      courses.append(row[0])
  return courses

In [5]:
courses = course_list()
print(courses[1:6])

['生産システムI', '特別講義\u3000国際紛争研究（外国語科目）', "特別講義\u3000Japan in Today's World（外国語科目）", '生態統計学', '商法演習']


### Q2.2
"keywor_list.csv" ファイルを1行ずつ順番に読み込み、その各行を要素とするリストを作成して返す vocab list 関数を完成さてください。作成されたリストは変数 vocab で受け取ります。以降の処理では、リスト vocab のインデックスをその要素(単語)のIDとして扱います。リスト vocab は以降の処理における語彙となります。

In [6]:
def vocab_list():
  keyword = []
  with open('data/keyword_list.csv') as f:
    reader = csv.reader(f)

    for row in reader:
      keyword.append(row[0])
  return keyword


In [7]:
vocab = vocab_list()
print(vocab[1:6])

['演習', '特殊', '講義', '科学', '研究']


### Q2.3
リスト courses と vocab を引数で受け取り、単語のID(リスト vocab のその単語のインデックス)をキー、その単語のDF (Document Frequency)を値とする辞書を作成して返す count_df 関数を作成してください。作成された辞書は変数 df で受け取ります。この場合、ある単語のDF値はその単語を講義名に含む講義数に対応します。

In [8]:
def count_df(courses, vocab):
    df ={}

    for i, word in enumerate(vocab):
        count=0
        for course in courses:
            if word in course: # 講義名に対象のwordが含まれているか
                count += 1
        df[i] = count
    return df

In [9]:
count = count_df(courses, vocab)
print(list(count.items())[:5])

[(0, 0), (1, 28), (2, 14), (3, 9), (4, 53)]


In [10]:
for i, v in list(count.items())[:5]:
  print(vocab[i], v)


keyword 0
演習 28
特殊 14
講義 9
科学 53


### Q2.4.1
リスト vocab の各単語を次元とする講義ベクトルを考えます。講義ベクトルの長さはリスト vocab の長さと等しく、リスト vocab のインデックス i の単語 vocab[i]が講義名に含まれる時、講義ベクトルの i 番目の要素は 1、含まれなければ0とします。
以下では、リスト courses と vocab を引数で受け取り、リスト courses の各講義のベクトルを行、リスト vocab の各単語を列とした NumPy の行列を作成して返すlec word matrix 関数を完成させてください。作成した講義-単語行列は、講義(行) の講義名に単語(列)が含まれていれば、その要素が1であるような行列です。

In [14]:
def lec_word_matrix(_courses, _vocab):
    # （講義数, 単語数）の行列を0で初期化
    lec_word = np.zeros((len(_courses), len(_vocab)))

    # 各講義・各単語についてチェック
    for i in range(len(_courses)):
        for j in range(len(_vocab)):
            if _vocab[j] in _courses[i]:
                lec_word[i][j] = 1

    return lec_word

In [15]:
np.sum(lec_word_matrix(courses, vocab))

np.float64(804.0)

In [17]:
# このデータでは合計は 804（header行を含む場合）
assert 804 == np.sum(lec_word_matrix(courses, vocab))